# NLP Assignment 5
## Token Classification: POS Tagging & Chunking


In [10]:

!pip install transformers datasets seqeval


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e80621afd7b752554915b1f1b4d49b335d5f5b76e811f04a11791b6e9c3e4a37
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [11]:
!pip install datasets transformers seqeval huggingface_hub


In [12]:
!wget https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train
!wget https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa
!wget https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb


--2026-04-04 15:25:10--  https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3281528 (3.1M) [text/plain]
Saving to: ‘eng.train’

eng.train           100%[===================>]   3.13M  --.-KB/s    in 0.05s   

2026-04-04 15:25:11 (65.0 MB/s) - ‘eng.train’ saved [3281528/3281528]

--2026-04-04 15:25:11--  https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 827012 (808K) [text/plain]

In [16]:
def read_conll(file_path):
    sentences = []

    with open(file_path, "r") as f:
        token_list, pos_list, chunk_list, ner_list = [], [], [], []

        for line in f:
            if line.strip() == "":
                if token_list:
                    if token_list[0] != "-DOCSTART-":
                        sentences.append({
                            "tokens": token_list,
                            "pos_tags": pos_list,
                            "chunk_tags": chunk_list,
                            "ner_tags": ner_list
                        })
                    token_list, pos_list, chunk_list, ner_list = [], [], [], []
            else:
                splits = line.split()
                token_list.append(splits[0])
                pos_list.append(splits[1])
                chunk_list.append(splits[2])
                ner_list.append(splits[3])

    return sentences


In [17]:
train_data = read_conll("eng.train")
val_data = read_conll("eng.testa")
test_data = read_conll("eng.testb")


In [18]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data),
})


In [19]:
def clean_tags(example):
    example["chunk_tags"] = [tag.strip() for tag in example["chunk_tags"]]
    return example

dataset = dataset.map(clean_tags)


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [20]:
unique_labels = set(tag for sent in dataset["train"]["chunk_tags"] for tag in sent)

label_list = sorted(list(unique_labels))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)


['B-ADJP', 'B-ADVP', 'B-NP', 'B-PP', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


In [21]:
def encode_labels(example):
    example["labels"] = [label2id[tag] for tag in example["chunk_tags"]]
    return example

dataset = dataset.map(encode_labels)


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [23]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [24]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [42]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
print(tokenized_datasets["train"][0])


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.'], 'chunk_tags': ['I-NP', 'I-VP', 'I-NP', 'I-NP', 'I-VP', 'I-VP', 'I-NP', 'I-NP', 'O'], 'ner_tags': ['I-ORG', 'O', 'I-MISC', 'O', 'O', 'O', 'I-MISC', 'O', 'O'], 'labels': [-100, 11, 15, 11, 11, 15, 15, 11, 11, 16, -100], 'input_ids': [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [27]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)


In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
)


In [39]:
!pip install seqeval
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [31]:
trainer.train()
trainer.evaluate()


Step,Training Loss
100,0.106531
200,0.096559
300,0.097122
400,0.106988
500,0.079894
600,0.102460
700,0.103421
800,0.081018
900,0.082762
1000,0.083856


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.18012861907482147,
 'eval_runtime': 10.3869,
 'eval_samples_per_second': 312.895,
 'eval_steps_per_second': 78.272,
 'epoch': 3.0}

In [40]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for pred, lab in zip(predictions, labels):
    temp_labels = []
    temp_preds = []

    for p_val, l_val in zip(pred, lab):
        if l_val != -100:
            temp_labels.append(id2label[l_val])
            temp_preds.append(id2label[p_val])

    true_labels.append(temp_labels)
    true_predictions.append(temp_preds)

print(classification_report(true_labels, true_predictions))


              precision    recall  f1-score   support

        ADJP       0.70      0.62      0.66       306
        ADVP       0.80      0.77      0.79       650
       CONJP       0.83      0.50      0.62        10
        INTJ       0.93      0.81      0.86        31
         LST       0.29      0.67      0.40         3
          NP       0.93      0.92      0.93     14644
          PP       0.97      0.97      0.97      4884
         PRT       0.77      0.82      0.80       147
        SBAR       0.86      0.83      0.84       366
          VP       0.92      0.91      0.91      4696

   micro avg       0.93      0.92      0.92     25737
   macro avg       0.80      0.78      0.78     25737
weighted avg       0.93      0.92      0.92     25737



In [41]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

print(nlp("John works at Google in California"))


[{'entity': 'I-NP', 'score': np.float32(0.9999088), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}, {'entity': 'I-VP', 'score': np.float32(0.9658889), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}, {'entity': 'I-PP', 'score': np.float32(0.9988367), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}, {'entity': 'I-NP', 'score': np.float32(0.9997029), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}, {'entity': 'I-PP', 'score': np.float32(0.9997538), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}, {'entity': 'I-NP', 'score': np.float32(0.9998871), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}]


# Task 7: Comparison

## POS Tagging vs Chunking

**POS Tagging → Grammar-level tagging (Easy)**

- Assigns grammatical roles (noun, verb, adjective)
- Works at word level
- Simpler and faster

**Chunking → Phrase-level grouping (Medium)**

- Groups words into phrases (NP, VP, PP)
- Works at phrase level
- More complex due to context understanding


# Task 8: Report / Blog


# Task 8: Report / Blog

## Introduction
In this assignment, a transformer-based model (BERT) was fine-tuned to perform token classification tasks, specifically Part-of-Speech (POS) Tagging and Chunking. These tasks are fundamental in Natural Language Processing (NLP) as they help machines understand the grammatical structure and phrase-level organization of sentences.

---

## Differences between POS Tagging and Chunking

Part-of-Speech (POS) Tagging assigns grammatical labels to individual words such as nouns, verbs, and adjectives. It is a word-level task and relatively simple.

Chunking, on the other hand, groups words into meaningful phrases such as noun phrases (NP), verb phrases (VP), and prepositional phrases (PP). It is a phrase-level task and more complex because it requires understanding relationships between words.

---

## Challenges Faced

- Dataset loading issues due to environment restrictions  
- Handling special tokens like `-DOCSTART-`  
- Label mapping errors due to inconsistent tags  
- Difficulty in aligning labels with subword tokens  
- Library version compatibility issues  

---

## Observations and Insights

- BERT achieved high performance with F1 score ≈ 0.92  
- Frequent classes (NP, VP) performed better than rare ones  
- Data preprocessing significantly improved results  
- Subword handling using `-100` was critical  
- Chunking is more complex than POS tagging  

---

## Conclusion

The project demonstrated that transformer models like BERT are highly effective for token classification tasks. Proper preprocessing and label alignment are essential for achieving high performance.
